**Prediction of Burnout in Remote Work Settings**

---

**Augmentation Notebook Version**


In [4]:
#############################
#       IMPORT LIBRARIES    #
#############################

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KernelDensity
from sklearn.model_selection import GridSearchCV

from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/ML Group Project CSCI 635')

from analysis.data_splits import train_test


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
#############################
#     DEFINE RANDOM SEED    #
#############################

seed = 11
rand_state = 11

np.random.seed(seed)
print("Seed set to:", seed)


Seed set to: 11


In [6]:
#############################
#     LOAD TRAINING DATA    #
#############################

def get_train_data():
    # Fetch training dataset from data_splits.py instead of direct csv
    X_train, X_test, y_train, y_test = train_test()

    sc = StandardScaler()
    X_train_scaled = sc.fit_transform(X_train)
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)

    return X_train_scaled, y_train.squeeze()

X, y = get_train_data()

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()


X shape: (1600, 11)
y shape: (1600,)


,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,app_switches,sleep_hours,task_completion,isolation_index,fatigue_score,day_type_Weekend
0,0.059892,-0.190374,-0.244904,1.097410,-0.431351,0.316649,-0.744093,-0.079895,-1.139238,-0.273703,-0.628445
1,-0.438256,-0.622754,-0.590752,1.097410,-0.431351,-1.321600,0.235483,0.891810,-1.139238,-0.359298,-0.628445
2,-1.239874,-1.070043,0.100944,-0.027778,-0.431351,-0.943542,0.134496,0.894133,-1.139238,-1.246781,-0.628445
3,0.145780,0.256915,-0.936600,-0.590372,-0.431351,0.022605,0.487951,0.386988,0.079198,-0.949452,1.591229
4,1.245142,0.962638,1.138489,0.534816,-0.431351,1.996905,-1.521695,-0.912231,1.297635,1.546313,-0.628445


In [7]:
#############################
#   SYNTHETIC DATA METHOD   #
#############################

def generate_synthetic_data(X, y, n_samples=2000):
    # Ensure y is 1-D
    y = y.squeeze()
    y_arr = y.values.reshape(-1)
    classes, counts = np.unique(y_arr, return_counts=True)

    kdes = {}  # Fit a different KDE model for each class

    # Fit a kernel density model using GridSearchCV to determine best bandwidth
    bandwidth_params = {'bandwidth': np.arange(0.01, 1, 0.05)}

    for cls in classes:
        X_cls = X[y_arr == cls]
        grid_search = GridSearchCV(KernelDensity(), bandwidth_params, cv=5)
        grid_search.fit(X_cls.values)
        kde = grid_search.best_estimator_
        kdes[cls] = kde

    # Maintain approximately original class distribution
    freqs = counts / counts.sum()
    samples_per_class = np.floor(freqs * n_samples).astype(int)

    remainder = n_samples - samples_per_class.sum()
    if remainder > 0:
        idxs = np.argsort(-counts)
        for i in range(remainder):
            samples_per_class[idxs[i % len(classes)]] += 1

    # Generate new synthetic samples
    new_X = []
    new_y = []

    for cls, n in zip(classes, samples_per_class):
        new_data = kdes[cls].sample(n, random_state=rand_state)
        new_X.append(new_data)
        new_y.append(np.full(n, cls))

    # Format the synthetic dataset
    X_synthset = np.vstack(new_X)
    y_synthset = np.concatenate(new_y)

    df_X_synth = pd.DataFrame(X_synthset, columns=X.columns)
    df_y_synth = pd.Series(y_synthset)

    return df_X_synth, df_y_synth


In [8]:
#############################
#   GENERATE SYNTHETIC DATA #
#############################

n_samples = 2000

df_X_synth, df_y_synth = generate_synthetic_data(X, y, n_samples)

print("Synthetic X shape:", df_X_synth.shape)
print("Synthetic y shape:", df_y_synth.shape)

df_X_synth.head()


Synthetic X shape: (2000, 11)
Synthetic y shape: (2000,)


,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,app_switches,sleep_hours,task_completion,isolation_index,fatigue_score,day_type_Weekend
0,-0.698164,-0.186150,0.763223,-0.973165,-0.293927,-1.828633,0.153969,-2.634567,-0.720773,-1.378130,-1.490043
1,-1.248939,-0.364158,-0.148354,-0.369748,-0.355925,-1.714569,0.825530,-1.514109,-1.438961,-0.683498,-0.391829
2,0.269760,-0.089368,-0.179965,0.065994,-0.863547,-0.968691,0.481465,0.367090,-0.643284,-0.457909,-0.524930
3,-0.333935,-0.601934,-1.020885,1.582837,-0.329034,-1.153279,0.712534,-0.421939,-0.179597,-0.644150,0.244323
4,-1.090191,-0.736081,0.029294,-0.335494,-0.597638,-0.491250,1.051610,1.221635,-0.926146,-0.446573,-0.430095


In [9]:
#############################
#   CHECK CLASS DISTRIBUTION #
#############################

print("Original class distribution:")
print(y.value_counts(normalize=True).sort_index())

print("\nSynthetic class distribution:")
print(df_y_synth.value_counts(normalize=True).sort_index())


Original class distribution:
burnout_risk
0    0.509375
1    0.421250
2    0.069375
Name: proportion, dtype: float64

Synthetic class distribution:
0    0.5095
1    0.4215
2    0.0690
Name: proportion, dtype: float64


In [10]:
#############################
#   AUGMENT TRAINING DATA   #
#############################

def augment_training_data(X, y, n_samples=2000):
    """
    Generate synthetic data and append it to the original training data.
    """
    df_X_synth, df_y_synth = generate_synthetic_data(X, y, n_samples)

    X_aug = pd.concat([X, df_X_synth], ignore_index=True)
    y_aug = pd.concat([y.squeeze(), df_y_synth], ignore_index=True)

    return X_aug, y_aug

X_aug, y_aug = augment_training_data(X, y, n_samples=2000)

print("Augmented X shape:", X_aug.shape)
print("Augmented y shape:", y_aug.shape)


Augmented X shape: (3600, 11)
Augmented y shape: (3600,)


In [11]:
#############################
#     PREVIEW OUTPUTS       #
#############################

print("Synthetic Features:")
display(df_X_synth.head())

print("\nSynthetic Labels:")
display(df_y_synth.head())

print("\nAugmented Features:")
display(X_aug.head())

print("\nAugmented Labels:")
display(y_aug.head())


Synthetic Features:


,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,app_switches,sleep_hours,task_completion,isolation_index,fatigue_score,day_type_Weekend
0,-0.698164,-0.186150,0.763223,-0.973165,-0.293927,-1.828633,0.153969,-2.634567,-0.720773,-1.378130,-1.490043
1,-1.248939,-0.364158,-0.148354,-0.369748,-0.355925,-1.714569,0.825530,-1.514109,-1.438961,-0.683498,-0.391829
2,0.269760,-0.089368,-0.179965,0.065994,-0.863547,-0.968691,0.481465,0.367090,-0.643284,-0.457909,-0.524930
3,-0.333935,-0.601934,-1.020885,1.582837,-0.329034,-1.153279,0.712534,-0.421939,-0.179597,-0.644150,0.244323
4,-1.090191,-0.736081,0.029294,-0.335494,-0.597638,-0.491250,1.051610,1.221635,-0.926146,-0.446573,-0.430095



Synthetic Labels:


,0
0,0
1,0
2,0
3,0
4,0



Augmented Features:


,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,app_switches,sleep_hours,task_completion,isolation_index,fatigue_score,day_type_Weekend
0,0.059892,-0.190374,-0.244904,1.097410,-0.431351,0.316649,-0.744093,-0.079895,-1.139238,-0.273703,-0.628445
1,-0.438256,-0.622754,-0.590752,1.097410,-0.431351,-1.321600,0.235483,0.891810,-1.139238,-0.359298,-0.628445
2,-1.239874,-1.070043,0.100944,-0.027778,-0.431351,-0.943542,0.134496,0.894133,-1.139238,-1.246781,-0.628445
3,0.145780,0.256915,-0.936600,-0.590372,-0.431351,0.022605,0.487951,0.386988,0.079198,-0.949452,1.591229
4,1.245142,0.962638,1.138489,0.534816,-0.431351,1.996905,-1.521695,-0.912231,1.297635,1.546313,-0.628445



Augmented Labels:


,0
0,0
1,0
2,0
3,0
4,1


In [12]:
#############################
#       SAVE AS CSV         #
#############################

save_path = "processed_data"
os.makedirs(save_path, exist_ok=True)

df_X_synth.to_csv(f"{save_path}/X_synth.csv", index=False)
df_y_synth.to_csv(f"{save_path}/y_synth.csv", index=False)
X_aug.to_csv(f"{save_path}/X_train_augmented.csv", index=False)
y_aug.to_csv(f"{save_path}/y_train_augmented.csv", index=False)

print(f"Saved augmentation files to: {save_path}")


Saved augmentation files to: processed_data


In [13]:
#############################
#        OPTIONAL MAIN      #
#############################

def main():
    n_samples = 2000

    X, y = get_train_data()
    if X is None or y is None:
        return

    df_X_synth, df_y_synth = generate_synthetic_data(X, y, n_samples)

    print(df_X_synth.head())
    print()
    print(df_y_synth.head())

if __name__ == "__main__":
    main()


   work_hours  screen_time_hours  meetings_count  breaks_taken  \
0   -0.698164          -0.186150        0.763223     -0.973165   
1   -1.248939          -0.364158       -0.148354     -0.369748   
2    0.269760          -0.089368       -0.179965      0.065994   
3   -0.333935          -0.601934       -1.020885      1.582837   
4   -1.090191          -0.736081        0.029294     -0.335494   

   after_hours_work  app_switches  sleep_hours  task_completion  \
0         -0.293927     -1.828633     0.153969        -2.634567   
1         -0.355925     -1.714569     0.825530        -1.514109   
2         -0.863547     -0.968691     0.481465         0.367090   
3         -0.329034     -1.153279     0.712534        -0.421939   
4         -0.597638     -0.491250     1.051610         1.221635   

   isolation_index  fatigue_score  day_type_Weekend  
0        -0.720773      -1.378130         -1.490043  
1        -1.438961      -0.683498         -0.391829  
2        -0.643284      -0.457909     